
# Pipeline OCR/VLM - Domiciliation des salaires étrangers

**Modèle : Qwen2.5-VL-7B-Instruct sur GPU Domino**

Ce notebook est inspiré de `pipeline ocr v12.ipynb`, avec une architecture dédiée aux dossiers de domiciliation :

- classification GPU des pages ;
- extraction spécialisée par type de document ;
- extraction KYC étendue, notamment le père et la mère ;
- conservation des valeurs brutes et normalisées ;
- génération d'un JSON par dossier ;
- export Excel avec les onglets `DOMICILIATIONS`, `PLANNING_TL`, `DOCUMENTS`, `CHAMPS_SOURCE`, `ERREURS` et `PARAMETRES` ;
- gestion des renouvellements et des mois scindés en `P1` / `P2` ;
- checkpoint après chaque dossier pour reprise après incident.

Le pipeline **n'effectue pas les contrôles réglementaires finaux**. Les règles métier et les décisions restent dans Alteryx.


## 1. Dépendances

In [ ]:

import sys
from importlib import metadata

REQUIRED_PACKAGES = {
    "torch": "2.0",
    "transformers": "4.45",
    "accelerate": "0.30",
    "PyMuPDF": "1.23",
    "Pillow": "9.0",
    "openpyxl": "3.1",
    "pandas": "1.5",
    "psutil": "5.9",
}

print("Python :", sys.version.replace("\n", " "))
print("\nPackages détectés :")
missing = []
for package_name, minimum in REQUIRED_PACKAGES.items():
    try:
        version = metadata.version(package_name)
        print(f"  {package_name:15s} {version:12s} | minimum conseillé {minimum}")
    except metadata.PackageNotFoundError:
        missing.append(package_name)
        print(f"  {package_name:15s} ABSENT")

if missing:
    raise RuntimeError(
        "Packages manquants : " + ", ".join(missing) +
        ". Installer uniquement ces packages dans l'environnement Domino."
    )

print("\n✅ Vérification des packages terminée sans modification de l'environnement")


## 2. Imports

In [ ]:

import gc
import hashlib
import json
import math
import re
import sys
import time
import calendar
from collections import defaultdict
from datetime import date, datetime, timedelta
from pathlib import Path

import fitz
import numpy as np
import pandas as pd
import psutil
import torch
from PIL import Image
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from transformers import AutoProcessor, AutoModelForVision2Seq

print("✅ Imports OK")
print("Python       :", sys.version.split()[0])
print("PyMuPDF     :", fitz.__doc__.splitlines()[0] if fitz.__doc__ else "chargé")
print("Torch       :", torch.__version__)
print("CUDA dispo  :", torch.cuda.is_available())
print("GPU         :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Aucun")


## 3. Configuration

In [ ]:
MODEL_PATH = '/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen2.5-VL-7B-Instruct/main'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PDF_ZOOM = 3.0
IMAGE_MAX_SIZE = 2400
BLANK_THRESHOLD = 0.995
GPU_BATCH_SIZE_CLASSIFICATION = 3
GPU_BATCH_SIZE_EXTRACTION = 2
MAX_NEW_TOKENS_CLASSIFICATION = 100
MAX_NEW_TOKENS_EXTRACTION = 1700
INPUT_DIR = Path('/mnt/data/domiciliations_in')
OUTPUT_DIR = Path('/mnt/data/domiciliations_out')
JSON_DIR = OUTPUT_DIR / 'json_dossiers'
LOG_PATH = OUTPUT_DIR / 'pipeline_domiciliations.log'
EXCEL_PATH = OUTPUT_DIR / f"domiciliations_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"
MASTER_JSON_PATH = OUTPUT_DIR / 'domiciliations_master.json'
PIPELINE_VERSION = "DOM_V5_EXTRACTION_BRUTE_PAR_PAGE"
PRORATA_MODE = 'CALENDAR_DAYS'
GENERER_MOIS_COMPLETS = True
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR.mkdir(parents=True, exist_ok=True)
pdfs = sorted(INPUT_DIR.glob('*.pdf'))
print(f'Device          : {DEVICE}')
print(f'PDFs détectés   : {len(pdfs)}')
print(f'Entrée          : {INPUT_DIR}')
print(f'Sortie          : {OUTPUT_DIR}')
print(f'Prorata retenu  : {PRORATA_MODE}')
CLASSIFICATION_THRESHOLD = 0.80


## 4. Chargement du modèle Qwen2.5-VL-7B

In [ ]:

if DEVICE != "cuda":
    raise RuntimeError("Ce pipeline nécessite un GPU CUDA.")

torch.backends.cuda.matmul.allow_tf32 = True
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print("Chargement du processor...")
t0 = time.time()
processor = AutoProcessor.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
)

print("Chargement du modèle...")
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_PATH,
    torch_dtype=DTYPE,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model.eval()
model.to(DEVICE)

print(f"✅ Modèle chargé en {time.time() - t0:.1f}s | dtype={DTYPE}")
print(f"VRAM allouée : {torch.cuda.memory_allocated() / 1e9:.2f} GB")


## 5. Utilitaires PDF, image et JSON

In [ ]:

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def resize_image(img, max_side=IMAGE_MAX_SIZE):
    w, h = img.size
    if max(w, h) <= max_side:
        return img
    ratio = max_side / max(w, h)
    return img.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)


def white_ratio(image):
    arr = np.array(image.convert("L"))
    return float((arr > 245).sum() / arr.size)


def is_blank(image, threshold=BLANK_THRESHOLD):
    return white_ratio(image) >= threshold


def pdf_to_pages(path, zoom=PDF_ZOOM):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"PDF introuvable : {path}")
    if path.stat().st_size == 0:
        raise ValueError(f"PDF vide : {path}")

    pages = []
    doc = fitz.open(str(path))
    try:
        page_count = int(doc.page_count)
        if page_count <= 0:
            raise ValueError(f"PyMuPDF ne détecte aucune page dans : {path.name}")

        matrix = fitz.Matrix(zoom, zoom)
        for i in range(page_count):
            page = doc.load_page(i)
            pix = page.get_pixmap(matrix=matrix, alpha=False)

            if pix.width <= 0 or pix.height <= 0 or not pix.samples:
                raise ValueError(
                    f"Rendu image vide : {path.name}, page {i + 1}"
                )

            img = Image.frombytes(
                "RGB",
                (pix.width, pix.height),
                pix.samples
            )
            img = resize_image(img)

            pages.append({
                "index": i,
                "page_num": i + 1,
                "image": img,
                "width": img.width,
                "height": img.height,
                "white_ratio": round(white_ratio(img), 6),
            })
    finally:
        doc.close()

    if len(pages) != page_count:
        raise RuntimeError(
            f"Conversion incomplète de {path.name}: "
            f"{len(pages)} image(s) pour {page_count} page(s)"
        )
    return pages


def parse_json_response(text):
    if not text:
        return {}

    clean = str(text).strip()
    clean = re.sub(r"^```(?:json)?", "", clean, flags=re.I).strip()
    clean = re.sub(r"```$", "", clean).strip()

    match = re.search(r"\{.*\}", clean, flags=re.S)
    if not match:
        return {}

    candidate = match.group(0)
    attempts = [
        candidate,
        re.sub(r",\s*([}\]])", r"\1", candidate),
    ]

    for attempt in attempts:
        try:
            parsed = json.loads(attempt)
            return parsed if isinstance(parsed, dict) else {}
        except Exception:
            continue
    return {}


def log(message):
    line = f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - {message}"
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")


print("✅ Utilitaires PDF/JSON OK")


## 6. Inférence GPU batch

In [ ]:

def ask_single(prompt, image, max_new_tokens):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ],
    }]

    text_in = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = processor(
        text=[text_in],
        images=[image],
        return_tensors="pt",
    ).to(DEVICE)

    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            top_p=1.0,
            repetition_penalty=1.0,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    generated = out[0][inputs["input_ids"].shape[1]:]
    text = processor.decode(
        generated,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )

    return {
        "text": text,
        "tokens_in": int(inputs["input_ids"].shape[1]),
        "tokens_out": int(len(generated)),
        "elapsed_s": round(time.time() - t0, 3),
    }


def ask_batch(prompt, images, max_new_tokens):
    if not images:
        return []
    if len(images) == 1:
        return [ask_single(prompt, images[0], max_new_tokens)]

    texts_in = []
    for image in images:
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }]
        texts_in.append(
            processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        )

    inputs = processor(
        text=texts_in,
        images=images,
        return_tensors="pt",
        padding=True,
    ).to(DEVICE)

    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            top_p=1.0,
            repetition_penalty=1.0,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    if out.shape[0] != len(images):
        raise RuntimeError(
            f"Réponses VLM incohérentes : {out.shape[0]} sortie(s) "
            f"pour {len(images)} image(s)"
        )

    elapsed = time.time() - t0
    input_width = inputs["input_ids"].shape[1]
    attention_mask = inputs.get("attention_mask")
    results = []

    for i in range(len(images)):
        generated = out[i][input_width:]
        text = processor.decode(
            generated,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )
        tokens_in = (
            int(attention_mask[i].sum().item())
            if attention_mask is not None
            else int(input_width)
        )
        results.append({
            "text": text,
            "tokens_in": tokens_in,
            "tokens_out": int(len(generated)),
            "elapsed_s": round(elapsed / len(images), 3),
        })

    return results


print("✅ Inférence single/batch OK")


## 7. Prompts de classification et d’extraction

In [ ]:

PROMPT_CLASSIFICATION = """
Analyse uniquement le titre, les en-têtes et la structure générale de cette page.

Classe la page dans exactement une seule catégorie :

- ENGAGEMENT_DOMICILIATION
- CONTRAT_TRAVAIL
- CONTRAT_SPECIFIQUE
- TITRE_TRAVAIL
- PERMIS_TRAVAIL_COUVERTURE
- AUTRE

Règles de classification :
- ENGAGEMENT_DOMICILIATION :
  le titre contient « ENGAGEMENT DE DOMICILIATION »
  ou « CONTRAT DES SALARIES ETRANGERS ».

- CONTRAT_TRAVAIL :
  le titre contient « CONTRAT DE TRAVAIL A DUREE DETERMINEE ».

- CONTRAT_SPECIFIQUE :
  le titre contient « CONTRAT DE TRAVAIL SPECIFIQUE
  A LA MAIN D’OEUVRE ETRANGERE ».

- TITRE_TRAVAIL :
  page bilingue contenant l'identité, le poste, l'employeur,
  les dates et la photo du travailleur.

- PERMIS_TRAVAIL_COUVERTURE :
  page contenant principalement « Permis de Travail »
  et « N° de Série ».

- AUTRE :
  aucun type ne correspond clairement.

Ne te base jamais uniquement sur le numéro de page.

Retourne uniquement ce JSON :
{
  "type_document": "TYPE",
  "confidence": 0.00,
  "titre_detecte": "TITRE BRUT OU null"
}
"""

COMMON_RAW_RULES = """
Tu analyses une seule page.

RÈGLES OBLIGATOIRES :
1. Extraire uniquement les champs demandés.
2. Pour chaque champ, rechercher le libellé indiqué.
3. Recopier uniquement la valeur située juste après le libellé :
   - sur la même ligne ;
   - ou immédiatement sur la ligne suivante si la valeur continue.
4. Conserver la valeur exactement comme elle apparaît :
   espaces, ponctuation, séparateurs, format de date et format de montant.
5. Ne corrige pas l'orthographe.
6. Ne normalise pas les dates.
7. Ne normalise pas les montants.
8. Ne sépare pas automatiquement le nom et le prénom.
9. Ne complète pas une valeur partiellement lisible.
10. N'utilise aucune valeur provenant d'une autre page.
11. Si le libellé est absent ou la valeur illisible, retourne null.
12. N'invente jamais une valeur.
13. Retourne uniquement un objet JSON valide, sans commentaire.
"""

PROMPT_ENGAGEMENT = COMMON_RAW_RULES + """
TYPE ATTENDU : ENGAGEMENT_DOMICILIATION

Extrais exactement les clés suivantes :

{
  "DOM_NOM_RAISON_SOCIAL_CLIENT": null,
  "DOM_COMPTE_LOCAL": null,
  "DOM_ADRESSE_CLIENT": null,
  "DOM_NUMERO_CONTRAT": null,
  "DOM_DUREE_CONTRAT_MOIS": null,
  "DOM_DATE_DEBUT_CONTRAT": null,
  "DOM_DATE_FIN_CONTRAT": null,
  "DOM_NOM_RAISON_SOCIAL_EMPLOYEUR": null,
  "DOM_ADRESSE_EMPLOYEUR": null,
  "DOM_SALAIRE_NET_MENSUEL": null,
  "DOM_PART_TRANSFERABLE": null,
  "DOM_TAUX_TRANSFERABLE": null,
  "DOM_MONTANT_TOTAL_DOMICILIE": null
}

Libellés et règles :

- DOM_NOM_RAISON_SOCIAL_CLIENT :
  valeur après « Nom et raison sociale ».

- DOM_COMPTE_LOCAL :
  valeur après « N de compte » ou « N° de compte ».

- DOM_ADRESSE_CLIENT :
  valeur après la première occurrence de « Adresse »
  dans la section « Identification du client ».

- DOM_NUMERO_CONTRAT :
  valeur après « Numéro du contrat ».

- DOM_DUREE_CONTRAT_MOIS :
  valeur après « Durée du contrat ».

- DOM_DATE_DEBUT_CONTRAT :
  valeur après « Date de début de contrat ».

- DOM_DATE_FIN_CONTRAT :
  valeur après « Date de fin de contrat ».

- DOM_NOM_RAISON_SOCIAL_EMPLOYEUR :
  valeur après « Nom et raison sociale de L’Employeur ».

- DOM_ADRESSE_EMPLOYEUR :
  valeur après « Adresse de L’Employeur ».
  Continuer sur la ligne suivante si l'adresse se poursuit.

- DOM_SALAIRE_NET_MENSUEL :
  valeur après « Salaire net mensuel ».

- DOM_PART_TRANSFERABLE :
  valeur après « Montant de la part transférable ».

- DOM_TAUX_TRANSFERABLE :
  valeur après « Pourcentage en regard du salaire net mensuel ».

- DOM_MONTANT_TOTAL_DOMICILIE :
  valeur après « Montant domicilié en DZD ».
"""

PROMPT_CONTRAT = COMMON_RAW_RULES + """
TYPE ATTENDU : CONTRAT_TRAVAIL

Extrais exactement les clés suivantes :

{
  "CTR_REFERENCE_DOCUMENT": null,
  "CTR_TYPE": null,
  "CTR_EMPLOYEUR": null,
  "CTR_ACTIVITE_EMPLOYEUR": null,
  "CTR_DUREE_MOIS": null,
  "CTR_DATE_DEBUT_CONTRAT": null,
  "CTR_POSTE": null,
  "CTR_NOM_PRENOM_TRAVAILLEUR": null,
  "CTR_PERE_NOM_PRENOM": null,
  "CTR_MERE_NOM_PRENOM": null,
  "CTR_NATIONALITE": null,
  "CTR_DATE_NAISSANCE": null,
  "CTR_LIEU_PAYS_NAISSANCE": null,
  "CTR_ADRESSE_ALGERIE": null,
  "CTR_QUALIFICATION": null,
  "CTR_NUMERO_PERMIS_TRAVAIL": null,
  "CTR_DATE_DELIVRANCE_PERMIS": null,
  "CTR_DATE_DEBUT_VALIDITE_PERMIS": null,
  "CTR_DATE_FIN_VALIDITE_PERMIS": null,
  "CTR_SALAIRE_BRUT": null,
  "CTR_SALAIRE_NET": null,
  "CTR_AFFILIATION_SS": null,
  "CTR_NUMERO_EMPLOYEUR": null,
  "CTR_DATE_SIGNATURE": null,
  "CTR_REFERENCE_DOMICILIATION": null,
  "CTR_SIGNATURE_TRAVAILLEUR_PRESENTE": null,
  "CTR_SIGNATURE_EMPLOYEUR_PRESENTE": null,
  "CTR_CACHET_EMPLOYEUR_PRESENT": null
}

Libellés et règles :

- CTR_REFERENCE_DOCUMENT :
  référence imprimée dans le coin supérieur gauche,
  par exemple « TR-3058 ».

- CTR_TYPE :
  titre complet du document.

- CTR_EMPLOYEUR :
  valeur après
  « au nom de l’employeur ci-après désigné : ».

- CTR_ACTIVITE_EMPLOYEUR :
  valeur après « Nature de l’activité : ».

- CTR_DUREE_MOIS :
  valeur après « pour une durée de : »
  et avant « à compter du ».

- CTR_DATE_DEBUT_CONTRAT :
  valeur après « à compter du : ».

- CTR_POSTE :
  valeur après « En qualité de : ».

- CTR_NOM_PRENOM_TRAVAILLEUR :
  valeur après « A (Mr/Mme) : ».

- CTR_PERE_NOM_PRENOM :
  valeur après « Fils de : »
  et avant « et de : ».

- CTR_MERE_NOM_PRENOM :
  valeur après « et de : ».

- CTR_NATIONALITE :
  valeur après « Nationalité : ».

- CTR_DATE_NAISSANCE :
  valeur après « Né(e) le : »
  et avant « à ».

- CTR_LIEU_PAYS_NAISSANCE :
  valeur après « à » sur la ligne de naissance.

- CTR_ADRESSE_ALGERIE :
  valeur après « Adresse en Algérie : ».

- CTR_QUALIFICATION :
  valeur après « Qualification professionnelle : ».

- CTR_NUMERO_PERMIS_TRAVAIL :
  valeur après « permis de travail N° ».

- CTR_DATE_DELIVRANCE_PERMIS :
  valeur après « Délivré le : ».

- CTR_DATE_DEBUT_VALIDITE_PERMIS :
  première date après « Valable du ».

- CTR_DATE_FIN_VALIDITE_PERMIS :
  date après « au » sur la même ligne.

- CTR_SALAIRE_BRUT :
  valeur après « Montant du salaire mensuel brut : ».

- CTR_SALAIRE_NET :
  valeur après « Montant du salaire mensuel net : ».

- CTR_AFFILIATION_SS :
  valeur après « Affiliation à la sécurité sociale : ».

- CTR_NUMERO_EMPLOYEUR :
  valeur après « Employeur : ».

- CTR_DATE_SIGNATURE :
  date après « Fait à : Bethioua, le ».

- CTR_REFERENCE_DOMICILIATION :
  dans le cachet « DOMICILIATION IMPORT »,
  recopier les cinq cases dans l'ordre
  et les séparer par « | ».
  Exemple : 271901|2026.1|40|00119|DZD

Contrôles visuels :
- CTR_SIGNATURE_TRAVAILLEUR_PRESENTE :
  true si un tracé manuscrit est visible directement sous
  « Signature du Travailleur Etranger », sinon false.

- CTR_SIGNATURE_EMPLOYEUR_PRESENTE :
  true si un tracé manuscrit est visible directement sous
  « Signature de l’Employeur », sinon false.

- CTR_CACHET_EMPLOYEUR_PRESENT :
  true si une empreinte de cachet est visible dans la zone
  « Signature de l’Employeur », sinon false.
"""

PROMPT_CONTRAT_SPECIFIQUE = COMMON_RAW_RULES + """
TYPE ATTENDU : CONTRAT_SPECIFIQUE

Extrais exactement les clés suivantes :

{
  "CTS_REFERENCE_DOCUMENT": null,
  "CTS_EMPLOYEUR": null,
  "CTS_ACTIVITE_EMPLOYEUR": null,
  "CTS_DUREE_MOIS": null,
  "CTS_DATE_DEBUT_CONTRAT": null,
  "CTS_POSTE": null,
  "CTS_NOM_PRENOM_TRAVAILLEUR": null,
  "CTS_PERE_NOM_PRENOM": null,
  "CTS_MERE_NOM_PRENOM": null,
  "CTS_NATIONALITE": null,
  "CTS_DATE_NAISSANCE": null,
  "CTS_LIEU_PAYS_NAISSANCE": null,
  "CTS_ADRESSE_ALGERIE": null,
  "CTS_QUALIFICATION": null,
  "CTS_NUMERO_PERMIS_TRAVAIL": null,
  "CTS_DATE_DELIVRANCE_PERMIS": null,
  "CTS_DATE_DEBUT_VALIDITE_PERMIS": null,
  "CTS_DATE_FIN_VALIDITE_PERMIS": null,
  "CTS_SALAIRE_NET": null,
  "CTS_PART_TRANSFERABLE": null,
  "CTS_PART_PAYABLE_DZD": null,
  "CTS_NUMERO_SS_PAYS_ORIGINE": null,
  "CTS_NUMERO_SS_ALGERIE": null,
  "CTS_DATE_DOCUMENT": null,
  "CTS_SIGNATURE_TRAVAILLEUR_PRESENTE": null,
  "CTS_SIGNATURE_EMPLOYEUR_PRESENTE": null,
  "CTS_CACHET_EMPLOYEUR_PRESENT": null
}

Libellés et règles :

- CTS_REFERENCE_DOCUMENT :
  référence imprimée dans le coin supérieur gauche.

- CTS_EMPLOYEUR :
  valeur après
  « au nom de l’employeur ci-après désigné : ».

- CTS_ACTIVITE_EMPLOYEUR :
  valeur après « Nature de l’activité : ».

- CTS_DUREE_MOIS :
  valeur après « pour une durée de : ».

- CTS_DATE_DEBUT_CONTRAT :
  valeur après « A compter du : ».

- CTS_POSTE :
  valeur après « en qualité de : ».

- CTS_NOM_PRENOM_TRAVAILLEUR :
  valeur après « A (Mr/Mme) : ».

- CTS_PERE_NOM_PRENOM :
  valeur après « Fils de : »
  et avant « et de : ».

- CTS_MERE_NOM_PRENOM :
  valeur après « et de : ».

- CTS_NATIONALITE :
  valeur après « Nationalité : ».

- CTS_DATE_NAISSANCE :
  valeur après « Né(e) le : »
  et avant « à ».

- CTS_LIEU_PAYS_NAISSANCE :
  valeur après « à » sur la ligne de naissance.

- CTS_ADRESSE_ALGERIE :
  valeur après « Adresse en Algérie : ».

- CTS_QUALIFICATION :
  valeur après « Qualification professionnelle : ».

- CTS_NUMERO_PERMIS_TRAVAIL :
  valeur après « permis de travail N° ».

- CTS_DATE_DELIVRANCE_PERMIS :
  valeur après « Délivré le : ».

- CTS_DATE_DEBUT_VALIDITE_PERMIS :
  première date après « valable du ».

- CTS_DATE_FIN_VALIDITE_PERMIS :
  date après « au » sur la même ligne.

- CTS_SALAIRE_NET :
  valeur après « Salaire mensuel de base net : ».

- CTS_PART_TRANSFERABLE :
  valeur après « La part transférable : ».

- CTS_PART_PAYABLE_DZD :
  valeur après « La part payable en dinars algérien : ».

- CTS_NUMERO_SS_PAYS_ORIGINE :
  valeur après « Dans le pays d’origine : ».

- CTS_NUMERO_SS_ALGERIE :
  valeur après « En Algérie : ».

- CTS_DATE_DOCUMENT :
  date après « Fait à : Bethioua, le ».

Contrôles visuels :
- CTS_SIGNATURE_TRAVAILLEUR_PRESENTE :
  true si un tracé manuscrit est visible sous
  « Signature du Travailleur Etranger ».

- CTS_SIGNATURE_EMPLOYEUR_PRESENTE :
  true si un tracé manuscrit est visible sous
  « Signature de l’Employeur ».

- CTS_CACHET_EMPLOYEUR_PRESENT :
  true si une empreinte de cachet est visible dans la zone employeur.
"""

PROMPT_TITRE_TRAVAIL = COMMON_RAW_RULES + """
TYPE ATTENDU : TITRE_TRAVAIL

Le document est bilingue arabe/français.
Utilise les libellés français.

Extrais exactement :

{
  "TTR_TYPE_DOCUMENT": null,
  "TTR_NUMERO": null,
  "TTR_POSTE": null,
  "TTR_DUREE": null,
  "TTR_DATE_DEBUT": null,
  "TTR_DATE_FIN": null,
  "TTR_NOM": null,
  "TTR_PRENOM": null,
  "TTR_DATE_NAISSANCE": null,
  "TTR_LIEU_NAISSANCE": null,
  "TTR_PAYS_NAISSANCE": null,
  "TTR_NATIONALITE": null,
  "TTR_QUALIFICATION": null,
  "TTR_EMPLOYEUR": null,
  "TTR_ADRESSE_EMPLOYEUR": null,
  "TTR_DATE_ENTREE_ALGERIE": null
}

Libellés et règles :

- TTR_TYPE_DOCUMENT :
  titre du document de travail.

- TTR_NUMERO :
  numéro visible dans la zone supérieure du titre,
  uniquement s'il existe un libellé ou un emplacement clairement associé.

- TTR_POSTE :
  valeur dans la zone supérieure gauche correspondant au poste.

- TTR_DUREE :
  valeur après « Durée ».

- TTR_DATE_DEBUT :
  valeur après « Du ».

- TTR_DATE_FIN :
  valeur après « Fin de travail ».

- TTR_NOM :
  valeur après « Nom ».

- TTR_PRENOM :
  valeur après « Prénom ».

- TTR_DATE_NAISSANCE :
  valeur après « Date de naissance ».

- TTR_LIEU_NAISSANCE :
  valeur après « Lieu de naissance ».

- TTR_PAYS_NAISSANCE :
  valeur après « Pays ».

- TTR_NATIONALITE :
  valeur après « Nationalité ».

- TTR_QUALIFICATION :
  valeur après « Qualification ».

- TTR_EMPLOYEUR :
  valeur après « Nom de l’organisme employeur ».

- TTR_ADRESSE_EMPLOYEUR :
  valeur après « Adresse de l’organisme employeur ».

- TTR_DATE_ENTREE_ALGERIE :
  valeur après « Date d’entrée en Algérie ».
"""

PROMPT_PERMIS_COUVERTURE = COMMON_RAW_RULES + """
TYPE ATTENDU : PERMIS_TRAVAIL_COUVERTURE

Extrais exactement :

{
  "PTR_NUMERO_SERIE": null
}

- PTR_NUMERO_SERIE :
  valeur après « N° de Série ».

Ne pas extraire les références légales imprimées à droite.
"""

PROMPTS_EXTRACTION = {
    "ENGAGEMENT_DOMICILIATION": PROMPT_ENGAGEMENT,
    "CONTRAT_TRAVAIL": PROMPT_CONTRAT,
    "CONTRAT_SPECIFIQUE": PROMPT_CONTRAT_SPECIFIQUE,
    "TITRE_TRAVAIL": PROMPT_TITRE_TRAVAIL,
    "PERMIS_TRAVAIL_COUVERTURE": PROMPT_PERMIS_COUVERTURE,
}

TYPES_VALIDES = set(PROMPTS_EXTRACTION) | {"AUTRE"}
print("✅ Prompts V5 bruts spécialisés chargés")


## 8. Normalisation technique

In [ ]:

NULL_VALUES = {"", "NULL", "NONE", "N/A", "NA", "NEANT", "NÉANT", "ILLISIBLE"}

def clean_raw_value(value):
    if value is None:
        return None
    if isinstance(value, bool):
        return value
    text = str(value).strip()
    return None if text.upper() in NULL_VALUES else text


def clean_raw_dict(data):
    if not isinstance(data, dict):
        return {}
    return {key: clean_raw_value(value) for key, value in data.items()}


def parse_date_for_planning(value):
    """Parsing technique réservé au planning, sans modifier la valeur brute."""
    if value is None:
        return None
    text = str(value).strip()
    for fmt in ("%d/%m/%Y", "%d-%m-%Y", "%Y-%m-%d"):
        try:
            return datetime.strptime(text, fmt).date()
        except ValueError:
            continue
    return None


def parse_amount_for_planning(value):
    """Parsing technique réservé au planning, sans modifier la valeur brute."""
    if value is None:
        return None
    text = str(value).replace("\xa0", " ").strip()
    text = re.sub(r"[^0-9,.\-]", "", text)
    if not text:
        return None
    if "," in text and "." in text:
        if text.rfind(",") > text.rfind("."):
            text = text.replace(".", "").replace(",", ".")
        else:
            text = text.replace(",", "")
    elif "," in text:
        text = text.replace(" ", "").replace(",", ".")
    else:
        text = text.replace(" ", "")
    try:
        return float(text)
    except ValueError:
        return None


print("✅ Helpers extraction brute et planning OK")


## 9. Consolidation des documents et informations KYC

In [ ]:

def build_page_row(pdf_name, page_record):
    row = {
        "FICHIER": pdf_name,
        "PAGE": page_record.get("page_num"),
        "TYPE_DOCUMENT": page_record.get("doc_type"),
        "TITRE_DETECTE": page_record.get("titre_detecte"),
        "CONFIANCE_CLASSIFICATION": page_record.get("classification_confidence"),
        "STATUT_EXTRACTION": page_record.get("extraction_status"),
        "ERREUR_EXTRACTION": page_record.get("extraction_error"),
        "TOKENS_CLASSIFICATION": (
            page_record.get("classification_tokens_in", 0)
            + page_record.get("classification_tokens_out", 0)
        ),
        "TOKENS_EXTRACTION": (
            page_record.get("extraction_tokens_in", 0)
            + page_record.get("extraction_tokens_out", 0)
        ),
    }
    row.update(page_record.get("raw_data") or {})
    return row


def month_segment_rows(pdf_name, engagement_data):
    start = parse_date_for_planning(
        engagement_data.get("DOM_DATE_DEBUT_CONTRAT")
    )
    end = parse_date_for_planning(
        engagement_data.get("DOM_DATE_FIN_CONTRAT")
    )
    if not start or not end or end < start:
        return []

    plafond = parse_amount_for_planning(
        engagement_data.get("DOM_PART_TRANSFERABLE")
    )
    salaire = parse_amount_for_planning(
        engagement_data.get("DOM_SALAIRE_NET_MENSUEL")
    )
    taux = engagement_data.get("DOM_TAUX_TRANSFERABLE")
    contract_number = engagement_data.get("DOM_NUMERO_CONTRAT")
    client = engagement_data.get("DOM_NOM_RAISON_SOCIAL_CLIENT")
    account = engagement_data.get("DOM_COMPTE_LOCAL")

    rows = []
    cursor = date(start.year, start.month, 1)

    while cursor <= end:
        last_day = calendar.monthrange(cursor.year, cursor.month)[1]
        month_end = date(cursor.year, cursor.month, last_day)
        segment_start = max(start, cursor)
        segment_end = min(end, month_end)

        if segment_start <= segment_end:
            if segment_start.day == 1 and segment_end.day == last_day:
                partie = ""
                periode = f"{cursor.year:04d}-{cursor.month:02d}"
            elif segment_start.day == 1:
                partie = "P1"
                periode = f"{cursor.year:04d}-{cursor.month:02d}P1"
            else:
                partie = "P2"
                periode = f"{cursor.year:04d}-{cursor.month:02d}P2"

            nb_jours = (segment_end - segment_start).days + 1
            montant_theorique = (
                round(plafond * nb_jours / last_day, 2)
                if plafond is not None else None
            )

            rows.append({
                "FICHIER": pdf_name,
                "DOM_NOM_RAISON_SOCIAL_CLIENT": client,
                "DOM_COMPTE_LOCAL": account,
                "DOM_NUMERO_CONTRAT": contract_number,
                "PERIODE_TL": periode,
                "MOIS_BASE": f"{cursor.year:04d}-{cursor.month:02d}",
                "PARTIE": partie or None,
                "DATE_DEBUT_SEGMENT": segment_start.isoformat(),
                "DATE_FIN_SEGMENT": segment_end.isoformat(),
                "NB_JOURS_SEGMENT": nb_jours,
                "NB_JOURS_MOIS": last_day,
                "SALAIRE_NET_REFERENCE_BRUT": engagement_data.get(
                    "DOM_SALAIRE_NET_MENSUEL"
                ),
                "TAUX_TRANSFERABLE_REFERENCE_BRUT": taux,
                "PLAFOND_MENSUEL_REFERENCE_BRUT": engagement_data.get(
                    "DOM_PART_TRANSFERABLE"
                ),
                "MONTANT_THEORIQUE_PRORATA": montant_theorique,
                "MONTANT_AUTORISE_SAISI": None,
            })

        if cursor.month == 12:
            cursor = date(cursor.year + 1, 1, 1)
        else:
            cursor = date(cursor.year, cursor.month + 1, 1)

    return rows


print("✅ Structure page/document et planning TL OK")


## 10. Gestion des mois scindés P1 / P2 et du planning TL

In [ ]:

# Test technique P1/P2
_demo = {
    "DOM_DATE_DEBUT_CONTRAT": "12/06/2025",
    "DOM_DATE_FIN_CONTRAT": "11/06/2026",
    "DOM_PART_TRANSFERABLE": "442 985,84",
    "DOM_SALAIRE_NET_MENSUEL": "466 300,88",
    "DOM_TAUX_TRANSFERABLE": "95%",
    "DOM_NUMERO_CONTRAT": "DEMO",
}
_demo_rows = month_segment_rows("demo.pdf", _demo)
assert _demo_rows[0]["PERIODE_TL"] == "2025-06P2"
assert _demo_rows[-1]["PERIODE_TL"] == "2026-06P1"
print("✅ Test planning P1/P2 réussi")


### Test de la règle P1 / P2 demandée

In [ ]:

print("Le planning est généré uniquement depuis la page ENGAGEMENT_DOMICILIATION.")


## 11. Classification et extraction d’un dossier

In [ ]:

def classify_pages(pages):
    records = []

    for start_idx in range(0, len(pages), GPU_BATCH_SIZE_CLASSIFICATION):
        batch = pages[start_idx:start_idx + GPU_BATCH_SIZE_CLASSIFICATION]
        outputs = ask_batch(
            PROMPT_CLASSIFICATION,
            [item["image"] for item in batch],
            MAX_NEW_TOKENS_CLASSIFICATION,
        )

        for page, output in zip(batch, outputs):
            parsed = parse_json_response(output["text"])
            doc_type = (
                parsed.get("type_document")
                or parsed.get("type")
                or "AUTRE"
            )
            confidence = parsed.get("confidence", 0)

            try:
                confidence = float(confidence or 0)
            except Exception:
                confidence = 0.0

            if doc_type not in TYPES_VALIDES:
                doc_type = "AUTRE"

            if confidence < CLASSIFICATION_THRESHOLD:
                doc_type = "AUTRE"

            records.append({
                "page_num": page["page_num"],
                "image": page["image"],
                "width": page["width"],
                "height": page["height"],
                "white_ratio": page["white_ratio"],
                "doc_type": doc_type,
                "titre_detecte": parsed.get("titre_detecte"),
                "classification_confidence": confidence,
                "classification_raw_text": output["text"],
                "classification_tokens_in": output["tokens_in"],
                "classification_tokens_out": output["tokens_out"],
                "classification_elapsed_s": output["elapsed_s"],
                "raw_data": {},
                "extraction_status": "NON_LANCEE",
                "extraction_error": None,
                "extraction_raw_text": None,
                "extraction_tokens_in": 0,
                "extraction_tokens_out": 0,
                "extraction_elapsed_s": 0.0,
            })

    return records


def extract_classified_pages(records):
    grouped = defaultdict(list)

    for record in records:
        if record["doc_type"] in PROMPTS_EXTRACTION:
            grouped[record["doc_type"]].append(record)
        else:
            record["extraction_status"] = "NON_APPLICABLE"

    for doc_type, group in grouped.items():
        prompt = PROMPTS_EXTRACTION[doc_type]

        for start_idx in range(0, len(group), GPU_BATCH_SIZE_EXTRACTION):
            batch = group[start_idx:start_idx + GPU_BATCH_SIZE_EXTRACTION]

            try:
                outputs = ask_batch(
                    prompt,
                    [item["image"] for item in batch],
                    MAX_NEW_TOKENS_EXTRACTION,
                )

                for record, output in zip(batch, outputs):
                    parsed = parse_json_response(output["text"])
                    record["raw_data"] = clean_raw_dict(parsed)
                    record["extraction_status"] = (
                        "OK" if parsed else "JSON_VIDE"
                    )
                    record["extraction_raw_text"] = output["text"]
                    record["extraction_tokens_in"] = output["tokens_in"]
                    record["extraction_tokens_out"] = output["tokens_out"]
                    record["extraction_elapsed_s"] = output["elapsed_s"]

            except Exception as exc:
                for record in batch:
                    record["extraction_status"] = "ERREUR"
                    record["extraction_error"] = repr(exc)

    return records


def process_pdf(pdf_path, verbose=True):
    t0 = time.time()

    if verbose:
        log(f"📁 {pdf_path.name}")

    pages = pdf_to_pages(pdf_path)
    if not pages:
        raise ValueError(f"Aucune page détectée dans {pdf_path.name}")

    records = classify_pages(pages)
    records = extract_classified_pages(records)

    page_rows = [
        build_page_row(pdf_path.name, record)
        for record in records
    ]

    engagement_data = {}
    for record in records:
        if record.get("doc_type") == "ENGAGEMENT_DOMICILIATION":
            engagement_data = record.get("raw_data") or {}
            break

    planning = month_segment_rows(pdf_path.name, engagement_data)

    tokens_in = sum(
        record.get("classification_tokens_in", 0)
        + record.get("extraction_tokens_in", 0)
        for record in records
    )
    tokens_out = sum(
        record.get("classification_tokens_out", 0)
        + record.get("extraction_tokens_out", 0)
        for record in records
    )

    dossier = {
        "source_file": pdf_path.name,
        "source_sha256": sha256_file(pdf_path),
        "pipeline_version": PIPELINE_VERSION,
        "stats": {
            "pages": len(pages),
            "tokens_in": tokens_in,
            "tokens_out": tokens_out,
            "tokens_total": tokens_in + tokens_out,
            "elapsed_s": round(time.time() - t0, 3),
        },
        "page_records": [
            {
                key: value
                for key, value in record.items()
                if key != "image"
            }
            for record in records
        ],
        "page_rows": page_rows,
        "planning_tl": planning,
    }

    checkpoint = (
        JSON_DIR
        / f"{pdf_path.stem}__{dossier['source_sha256'][:12]}.json"
    )
    with open(checkpoint, "w", encoding="utf-8") as f:
        json.dump(
            dossier,
            f,
            ensure_ascii=False,
            indent=2,
            default=str,
        )

    if verbose:
        log(
            f"✅ {pdf_path.name} | pages={len(pages)} "
            f"| tokens={tokens_in + tokens_out} "
            f"| planning={len(planning)}"
        )

    return dossier


print("✅ Classification puis extraction spécialisée V5 OK")


## 12. Export Excel compatible Alteryx

In [ ]:

def ordered_columns(rows):
    base = [
        "FICHIER",
        "PAGE",
        "TYPE_DOCUMENT",
        "TITRE_DETECTE",
        "CONFIANCE_CLASSIFICATION",
        "STATUT_EXTRACTION",
        "ERREUR_EXTRACTION",
        "TOKENS_CLASSIFICATION",
        "TOKENS_EXTRACTION",
    ]
    cols = set()
    for row in rows:
        cols.update(row.keys())
    ordered = [column for column in base if column in cols]
    ordered.extend(sorted(cols - set(ordered)))
    return ordered


def sheet_from_rows(wb, title, rows, columns=None):
    ws = wb.create_sheet(title)

    if columns is None:
        columns = ordered_columns(rows) if rows else []

    if not columns:
        ws["A1"] = "Aucune donnée"
        return ws

    header_fill = PatternFill("solid", fgColor="1F4E78")
    header_font = Font(color="FFFFFF", bold=True, name="Arial", size=9)

    for col_idx, name in enumerate(columns, start=1):
        cell = ws.cell(row=1, column=col_idx, value=name)
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True,
        )

    for row_idx, row in enumerate(rows, start=2):
        for col_idx, name in enumerate(columns, start=1):
            value = row.get(name)
            if isinstance(value, (dict, list)):
                value = json.dumps(value, ensure_ascii=False)
            ws.cell(row=row_idx, column=col_idx, value=value)

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    ws.row_dimensions[1].height = 36

    for idx, name in enumerate(columns, start=1):
        if "ADRESSE" in name:
            width = 38
        elif "NOM" in name or "EMPLOYEUR" in name:
            width = 30
        elif "DATE" in name or "PERIODE" in name:
            width = 18
        elif "MONTANT" in name or "SALAIRE" in name or "PART_" in name:
            width = 22
        else:
            width = min(32, max(14, len(name) + 2))
        ws.column_dimensions[get_column_letter(idx)].width = width

    return ws


def build_long_raw_rows(dossiers):
    rows = []
    for dossier in dossiers:
        for page in dossier.get("page_records", []):
            for field_name, raw_value in (page.get("raw_data") or {}).items():
                rows.append({
                    "FICHIER": dossier.get("source_file"),
                    "PAGE": page.get("page_num"),
                    "TYPE_DOCUMENT": page.get("doc_type"),
                    "CHAMP": field_name,
                    "VALEUR_BRUTE": raw_value,
                    "CONFIANCE_CLASSIFICATION": page.get(
                        "classification_confidence"
                    ),
                    "STATUT_EXTRACTION": page.get("extraction_status"),
                })
    return rows


def create_excel(excel_path, dossiers, errors):
    page_rows = []
    planning_rows = []

    for dossier in dossiers:
        page_rows.extend(dossier.get("page_rows", []))
        planning_rows.extend(dossier.get("planning_tl", []))

    raw_rows = build_long_raw_rows(dossiers)

    params_rows = [
        {"PARAMETRE": "MODEL_PATH", "VALEUR": MODEL_PATH},
        {"PARAMETRE": "PIPELINE_VERSION", "VALEUR": PIPELINE_VERSION},
        {"PARAMETRE": "CLASSIFICATION_THRESHOLD", "VALEUR": CLASSIFICATION_THRESHOLD},
        {"PARAMETRE": "PDF_ZOOM", "VALEUR": PDF_ZOOM},
        {"PARAMETRE": "IMAGE_MAX_SIZE", "VALEUR": IMAGE_MAX_SIZE},
        {"PARAMETRE": "GPU_BATCH_SIZE_CLASSIFICATION", "VALEUR": GPU_BATCH_SIZE_CLASSIFICATION},
        {"PARAMETRE": "GPU_BATCH_SIZE_EXTRACTION", "VALEUR": GPU_BATCH_SIZE_EXTRACTION},
    ]

    wb = Workbook()
    wb.remove(wb.active)

    sheet_from_rows(wb, "PAGES_DOCUMENTS", page_rows)
    sheet_from_rows(
        wb,
        "EXTRACTION_BRUTE",
        raw_rows,
        [
            "FICHIER",
            "PAGE",
            "TYPE_DOCUMENT",
            "CHAMP",
            "VALEUR_BRUTE",
            "CONFIANCE_CLASSIFICATION",
            "STATUT_EXTRACTION",
        ],
    )
    sheet_from_rows(wb, "PLANNING_TL", planning_rows)
    sheet_from_rows(wb, "ERREURS", errors)
    sheet_from_rows(
        wb,
        "PARAMETRES",
        params_rows,
        ["PARAMETRE", "VALEUR"],
    )

    wb.save(excel_path)
    print(
        f"✅ Excel créé : {excel_path} "
        f"| pages={len(page_rows)} "
        f"| champs bruts={len(raw_rows)} "
        f"| planning={len(planning_rows)}"
    )


print("✅ Export Excel V5 par page/document OK")


In [ ]:

print(f"Nombre de PDF détectés : {len(pdfs)}")
if not pdfs:
    raise RuntimeError(
        f"Aucun PDF trouvé dans {INPUT_DIR}. "
        "Déposer les dossiers de domiciliation dans ce répertoire."
    )

diagnostic_errors = []
total_pages = 0

for p in pdfs:
    try:
        with fitz.open(str(p)) as doc:
            page_count = int(doc.page_count)
        total_pages += page_count
        print(
            f"{p.name} -> pages={page_count}, "
            f"taille={p.stat().st_size:,} octets"
        )
        if page_count <= 0:
            diagnostic_errors.append(f"{p.name}: 0 page")
    except Exception as exc:
        diagnostic_errors.append(f"{p.name}: {exc!r}")

if diagnostic_errors:
    raise RuntimeError(
        "Diagnostic PDF en erreur :\n- " + "\n- ".join(diagnostic_errors)
    )

print(f"✅ Diagnostic PDF : {len(pdfs)} fichier(s), {total_pages} page(s)")

# Nettoyage des anciens checkpoints vides uniquement.
removed = 0
for json_file in JSON_DIR.glob("*.json"):
    try:
        dossier = json.loads(json_file.read_text(encoding="utf-8"))
        pages_checkpoint = int(
            dossier.get("stats", {}).get("pages", 0) or 0
        )
        records_checkpoint = dossier.get("page_records") or []
        if pages_checkpoint <= 0 or not records_checkpoint:
            json_file.unlink()
            removed += 1
            print(f"Checkpoint vide supprimé : {json_file.name}")
    except Exception:
        # Un checkpoint illisible ne doit pas être repris.
        json_file.unlink()
        removed += 1
        print(f"Checkpoint illisible supprimé : {json_file.name}")

print(f"Checkpoints supprimés : {removed}")


In [ ]:

# Test technique sur le premier PDF avant le traitement complet.
_test_pdf = pdfs[0]
_test_pages = pdf_to_pages(_test_pdf)

print(
    f"Test conversion : {_test_pdf.name} -> "
    f"{len(_test_pages)} page(s)"
)
print(
    "Dimensions première page :",
    _test_pages[0]["width"],
    "x",
    _test_pages[0]["height"],
)
print(
    "Ratio blanc première page :",
    _test_pages[0]["white_ratio"],
)

if len(_test_pages) == 0:
    raise RuntimeError("Le test de conversion PDF a retourné zéro page.")

# Libération immédiate des images du test.
del _test_pages
gc.collect()
print("✅ Test de conversion PDF réussi")


## 13. Exécution complète avec reprise automatique

In [ ]:

ram_free = psutil.virtual_memory().available / 1_000_000_000
log(f"RAM libre : {ram_free:.1f} GB")

existing_by_hash = {}

for json_file in JSON_DIR.glob("*.json"):
    try:
        with open(json_file, encoding="utf-8") as f:
            dossier = json.load(f)

        if dossier.get("pipeline_version") != PIPELINE_VERSION:
            continue

        file_hash = dossier.get("source_sha256")
        if file_hash:
            existing_by_hash[file_hash] = dossier

    except Exception as exc:
        log(f"⚠️ Checkpoint illisible {json_file.name}: {exc}")

all_dossiers = []
errors = []

for position, pdf_path in enumerate(pdfs, start=1):
    try:
        file_hash = sha256_file(pdf_path)

        if file_hash in existing_by_hash:
            dossier = existing_by_hash[file_hash]

            if (
                dossier.get("page_records")
                and int(dossier.get("stats", {}).get("pages", 0) or 0) > 0
            ):
                all_dossiers.append(dossier)
                log(
                    f"[{position}/{len(pdfs)}] "
                    f"↩️ Reprise checkpoint V5 : {pdf_path.name}"
                )
                continue

        dossier = process_pdf(pdf_path)
        all_dossiers.append(dossier)

    except Exception as exc:
        errors.append({
            "FICHIER": pdf_path.name,
            "ETAPE": "PROCESS_PDF",
            "ERREUR": repr(exc),
            "DATE": datetime.now().isoformat(timespec="seconds"),
        })
        log(
            f"[{position}/{len(pdfs)}] "
            f"❌ {pdf_path.name}: {exc}"
        )

    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

create_excel(EXCEL_PATH, all_dossiers, errors)

master_payload = {
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "pipeline_version": PIPELINE_VERSION,
    "dossiers": all_dossiers,
    "errors": errors,
}

with open(MASTER_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(
        master_payload,
        f,
        ensure_ascii=False,
        indent=2,
        default=str,
    )

log(f"JSON maître : {MASTER_JSON_PATH}")
log(
    f"✅ Pipeline terminé "
    f"| dossiers={len(all_dossiers)} "
    f"| erreurs={len(errors)}"
)



## 14. Lecture des résultats

- `domiciliations_master.json` : référentiel consolidé avec les liens de renouvellement et tout le planning.
- `DOMICILIATIONS` : une ligne par contrat/domiciliation, avec informations KYC, père, mère, montants contractuels, permis, données brutes par document et montants P1/P2.
- `PLANNING_TL` : une ligne par période de transfert autorisée. Les mois partiels sont nommés, par exemple, `2025-06P2` et `2026-06P1`.
- `PLN_MONTANT_AUTORISE_SAISI` : colonne volontairement vide destinée à être complétée/validée avant les contrôles mensuels.
- `DOCUMENTS` : classification et extraction page par page.
- `CHAMPS_SOURCE` : traçabilité longue de toutes les valeurs brutes et normalisées.
- `ERREURS` : erreurs techniques.
- `PARAMETRES` : version du modèle et règles de génération.


In [ ]:

if EXCEL_PATH.exists():
    df_pages = pd.read_excel(
        EXCEL_PATH,
        sheet_name="PAGES_DOCUMENTS",
    )
    df_raw = pd.read_excel(
        EXCEL_PATH,
        sheet_name="EXTRACTION_BRUTE",
    )
    df_plan = pd.read_excel(
        EXCEL_PATH,
        sheet_name="PLANNING_TL",
    )

    print(f"Pages/documents : {len(df_pages)}")
    print(f"Champs bruts    : {len(df_raw)}")
    print(f"Périodes TL     : {len(df_plan)}")

    display(df_pages.head(10))
    display(df_raw.head(30))
    display(df_plan.head(15))
else:
    print("Le fichier Excel n'a pas encore été généré.")
